# Deteccion de Muelas del Juicio — Entrega 2: Dataset en PyTorch

**Materia:** Redes Neuronales  
**Docente:** Ing. Pablo Marinozi  
**Dataset:** DENTEX — MICCAI 2023 ([arXiv:2305.19112](https://arxiv.org/abs/2305.19112))  
**Repo:** https://github.com/JulianOliveraBalls/dentex-wisdom-teeth

Notebook independiente — descarga el dataset desde Hugging Face automaticamente.

---

| Seccion | Contenido | Consigna |
|---------|-----------|---------|
| 1 | Setup: entorno, paths, dependencias | — |
| 2 | Descarga del dataset desde HuggingFace | f |
| 3 | Estructura del dataset y acceso a imagenes | a |
| 4 | Clase Dataset de PyTorch | a |
| 5 | Particionado y DataLoaders | b |
| 6 | Preprocesamiento y normalizacion | c |
| 7 | Combinaciones de Data Augmentation | d |
| 8 | Expansion del dataset con cuadrantes | d |
| 9 | Verificacion final del batch | e |
| 10 | Exportar CSVs para GitHub | f |
| 11 | HierarchicalDet: arquitectura del paper | — |
| 12 | Experimentos de fine-tuning | — |
| 13 | Conclusiones | — |

## Seccion 1 — Setup

In [ ]:
# Celda 0 — Correr PRIMERO y reiniciar sesion antes de continuar
# Solo necesaria la primera vez en una sesion nueva de Colab
!pip install torch==2.3.0 torchvision==0.18.0 --upgrade -q
!pip install ultralytics scikit-learn huggingface_hub -q
print('Reinicia la sesion: Entorno de ejecucion > Reiniciar sesion')

In [ ]:
import subprocess, sys, json, shutil, random, warnings, os, zipfile
from pathlib import Path
from collections import Counter, defaultdict
warnings.filterwarnings('ignore')

# Detectar entorno
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f'Entorno: {"Google Colab" if IN_COLAB else "Local"}')

# Paths segun entorno
if IN_COLAB:
    BASE_DIR = Path('/content')
else:
    BASE_DIR = Path.cwd().parent if Path.cwd().name == 'dev' else Path.cwd()

DATA_DIR    = BASE_DIR / 'data'
DATASET_DIR = DATA_DIR / 'raw' / 'dentex_raw'
YOLO_DIR    = DATA_DIR / 'processed' / 'yolo_dataset'
AUG_DIR     = DATA_DIR / 'processed' / 'yolo_augmented'  # dataset expandido con cuadrantes
SPLITS_DIR  = DATA_DIR
OUTPUTS_DIR = DATA_DIR / 'outputs'

for d in [DATASET_DIR, YOLO_DIR, AUG_DIR, OUTPUTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'DATA_DIR:    {DATA_DIR}')
print(f'DATASET_DIR: {DATASET_DIR}')
print(f'YOLO_DIR:    {YOLO_DIR}')

In [ ]:
# GPU check
try:
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True, timeout=5)
    print('GPU detectada' if result.returncode == 0 else 'Sin GPU')
except (FileNotFoundError, subprocess.TimeoutExpired):
    print('Sin GPU — OK para preparacion de datos')

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from sklearn.model_selection import train_test_split

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

print(f'PyTorch: {torch.__version__}')
print(f'CUDA:    {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device:  {torch.cuda.get_device_name(0)}')

def is_real_file(p):
    try:
        return p.resolve().exists() and p.stat().st_size > 0
    except (OSError, FileNotFoundError):
        return False

def log(msg, level='INFO'):
    icons = {'INFO':'[INFO]','OK':'[OK]  ','WARN':'[WARN]','ERR':'[ERR] ','DATA':'[DATA]'}
    print(f'{icons.get(level,"[INFO]")} {msg}')

## Seccion 2 — Descarga del dataset desde HuggingFace

El repo DENTEX tiene los datos organizados en ZIPs:
- `DENTEX/training_data.zip` — 705 imagenes con anotaciones (el que usamos)
- `DENTEX/validation_data.zip` — 50 imagenes de validacion oficial
- `DENTEX/test_data.zip` — 250 imagenes de test (sin anotaciones publicas)

Se descarga automaticamente si no existe en disco.

In [ ]:
from huggingface_hub import hf_hub_download

def descargar_y_extraer(filename, desc):
    log(f'Descargando {desc}...', 'WARN')
    zip_path = hf_hub_download(
        repo_id='ibrahimhamamci/DENTEX',
        repo_type='dataset',
        filename=filename,
        local_dir=str(DATASET_DIR),
    )
    log(f'Descomprimiendo {desc}...', 'INFO')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(str(DATASET_DIR / 'DENTEX'))
    log(f'{desc} listo', 'OK')

# Verificar si ya esta descargado
json_files = [f for f in DATASET_DIR.rglob('*.json') if is_real_file(f)]
train_ok = any('disease' in f.name.lower() for f in json_files)

if train_ok:
    log(f'Dataset encontrado — {len(json_files)} JSONs', 'OK')
else:
    descargar_y_extraer('DENTEX/training_data.zip', 'training_data (~3GB)')
    descargar_y_extraer('DENTEX/validation_data.zip', 'validation_data')
    json_files = [f for f in DATASET_DIR.rglob('*.json') if is_real_file(f)]
    log(f'Descarga completa — {len(json_files)} JSONs', 'OK')

# Detectar JSON principal (dataset c: quadrant-enumeration-disease)
TRAIN_JSON = next((j for j in json_files if 'disease' in j.name.lower()), None)
if TRAIN_JSON is None:
    raise FileNotFoundError('JSON dataset (c) no encontrado')
log(f'JSON principal: {TRAIN_JSON.name}', 'OK')

# Detectar carpeta de imagenes
IMG_DIR_C = next(
    (c for c in [TRAIN_JSON.parent/'xrays', TRAIN_JSON.parent.parent/'xrays']
     if c.exists() and c.is_dir()),
    TRAIN_JSON.parent / 'xrays'
)
imgs_reales = [p for p in list(IMG_DIR_C.glob('*.png'))+list(IMG_DIR_C.glob('*.jpg'))
               if is_real_file(p)]
log(f'Imagenes en disco: {len(imgs_reales)}', 'OK')

## Seccion 3 — Estructura del dataset y acceso a imagenes

### Formato COCO
Las anotaciones siguen el formato COCO JSON:
```
{
  "images": [{"id": 1, "file_name": "train_1.png", "width": 2744, "height": 1316}, ...],
  "annotations": [{"image_id": 1, "bbox": [x, y, w, h], "category_id_2": 7, ...}, ...],
  "categories_1": [...],  "categories_2": [...],  "categories_3": [...]
}
```

### Sistema FDI
- `category_id_1` — cuadrante (1-4, donde 0 = sin asignar pero valido)
- `category_id_2` — posicion del diente (1-8). **7 = tercer molar = muela del juicio**
- `category_id_3` — diagnostico. **0 = impacted**, 1=caries, 2=deep caries, 3=periapical

### Acceso a las imagenes
Las imagenes estan en `training_data/quadrant-enumeration-disease/xrays/`  
Formato PNG, escala de grises convertida a RGB (3 canales identicos).

In [ ]:
with open(TRAIN_JSON) as f:
    data_c = json.load(f)

id_to_image_c = {img['id']: img for img in data_c['images']}

# Filtrar muelas del juicio (cat2==7)
wisdom_c = defaultdict(list)
for ann in data_c['annotations']:
    if ann.get('category_id_2') == 7:
        wisdom_c[ann['image_id']].append(ann)

def classify_image(img_id):
    return 'impacted' if any(a.get('category_id_3')==0 for a in wisdom_c[img_id]) else 'erupted'

img_labels_c = {img_id: classify_image(img_id) for img_id in wisdom_c}
label_counts  = Counter(img_labels_c.values())
total_c       = sum(label_counts.values())

log(f'Total imagenes en JSON: {len(data_c["images"])}', 'DATA')
log(f'Con muela del juicio (cat2==7): {total_c}', 'DATA')
log(f'  erupted:  {label_counts["erupted"]} ({label_counts["erupted"]/total_c*100:.1f}%)', 'DATA')
log(f'  impacted: {label_counts["impacted"]} ({label_counts["impacted"]/total_c*100:.1f}%)', 'DATA')
log(f'  ratio:    {max(label_counts.values())/min(label_counts.values()):.2f}x', 'DATA')

# Ejemplo de anotacion
ann_ej = next(ann for anns in wisdom_c.values() for ann in anns)
log(f'Ejemplo de anotacion:', 'INFO')
log(f'  bbox (COCO):      {[round(v) for v in ann_ej["bbox"]]}  [x_tl, y_tl, w, h]', 'DATA')
log(f'  category_id_2:    {ann_ej.get("category_id_2")}  (7 = muela del juicio)', 'DATA')
log(f'  category_id_3:    {ann_ej.get("category_id_3")}  (0=impacted, otro=erupted)', 'DATA')

## Seccion 4 — Clase Dataset de PyTorch

**Por que no ImageFolder:**
`ImageFolder` asume una imagen = una etiqueta global (clasificacion).  
Nuestro problema es deteccion de objetos — cada imagen puede tener multiples  
bounding boxes con distintas clases simultaneamente. Necesitamos Dataset personalizado.

La clase `DentexDataset` lee los archivos `.txt` en formato YOLO y los convierte  
a tensores de PyTorch con boxes normalizadas y etiquetas.

In [ ]:
class DentexDataset(Dataset):
    """
    Dataset PyTorch para DENTEX — deteccion de muelas del juicio.
    Lee imagenes y labels en formato YOLO desde disco.

    Item devuelto:
        image  (Tensor [C,H,W]): imagen normalizada
        target (dict):
            boxes    (Tensor [N,4]): coords YOLO normalizadas [xc,yc,w,h]
            labels   (Tensor [N]):   0=erupted, 1=impacted
            image_id (str):          nombre del archivo
    """
    CLASS_NAMES  = {0: 'erupted', 1: 'impacted'}
    CLASS_COLORS = {0: '#44AA44', 1: '#FF4444'}

    def __init__(self, split, yolo_dir, transform=None):
        assert split in ('train', 'val', 'test'), f'Split invalido: {split}'
        self.split     = split
        self.yolo_dir  = Path(yolo_dir)
        self.transform = transform
        self.img_dir   = self.yolo_dir / 'images' / split
        self.lbl_dir   = self.yolo_dir / 'labels' / split

        self.samples = [
            (p, self.lbl_dir / (p.stem + '.txt'))
            for p in sorted(list(self.img_dir.glob('*.png')) + list(self.img_dir.glob('*.jpg')))
            if (self.lbl_dir / (p.stem + '.txt')).exists()
        ]

        self.class_counts = Counter()
        for _, lp in self.samples:
            for line in lp.read_text().strip().split('\n'):
                if line: self.class_counts[int(line.split()[0])] += 1

        log(f'DentexDataset [{split}]: {len(self.samples)} imgs | '
            f'erupted={self.class_counts[0]} impacted={self.class_counts[1]}', 'OK')

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        img_path, lbl_path = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        boxes, labels = [], []
        for line in lbl_path.read_text().strip().split('\n'):
            if line:
                p = line.split()
                labels.append(int(p[0]))
                boxes.append(list(map(float, p[1:])))
        if self.transform:
            image = self.transform(image)
        return image, {
            'boxes':    torch.tensor(boxes,  dtype=torch.float32) if boxes  else torch.zeros((0,4)),
            'labels':   torch.tensor(labels, dtype=torch.long)    if labels else torch.zeros(0, dtype=torch.long),
            'image_id': img_path.stem,
        }

    def get_image_raw(self, idx):
        return Image.open(self.samples[idx][0]).convert('RGB')

log('Clase DentexDataset definida', 'OK')

## Seccion 5 — Particionado y DataLoaders

### Criterio de particion
- **Estratificado**: mantiene la misma proporcion erupted/impacted en los 3 splits
- **Proporciones**: 70% train / 5% val / 25% test
- **seed=42**: particion reproducible — mismas imagenes siempre
- **Test separado**: nunca se usa para ajustar hiperparametros

### Justificacion del tamano de val
Con solo 22 imagenes en val, 1 imagen = 4.5% de variacion en las metricas.  
Es ruidoso pero inevitable dado el tamano del dataset. El numero clave es **test (110 imgs)**.

### DataLoaders
- `batch_size=8`: imagenes de ~2800px redimensionadas a 640px
- `shuffle=True` solo en train
- `collate_fn` personalizado: cada imagen tiene distinto numero de boxes

In [ ]:
def generar_split_yolo(yolo_out_dir, img_ids, labels, wisdom, id_to_img, img_dir_src, suffix=''):
    """Genera split YOLO en yolo_out_dir con las imagenes e ids dados."""
    ids_tr, ids_tmp, _, y_tmp = train_test_split(
        img_ids, labels, test_size=0.30, stratify=labels, random_state=42)
    ids_val, ids_test, _, _ = train_test_split(
        ids_tmp, y_tmp, test_size=0.833, stratify=y_tmp, random_state=42)

    splits = {'train': ids_tr, 'val': ids_val, 'test': ids_test}
    errors = 0

    for split, ids in splits.items():
        (yolo_out_dir/'images'/split).mkdir(parents=True, exist_ok=True)
        (yolo_out_dir/'labels'/split).mkdir(parents=True, exist_ok=True)
        for img_id in ids:
            info     = id_to_img[img_id]
            src_path = img_dir_src / Path(info['file_name']).name
            if not is_real_file(src_path): errors += 1; continue
            pil  = Image.open(src_path)
            W, H = pil.size
            shutil.copy2(src_path, yolo_out_dir/'images'/split/src_path.name)
            lines = []
            for ann in wisdom[img_id]:
                x, y, bw, bh = ann['bbox']
                xc = min(max((x+bw/2)/W, 0.0), 1.0)
                yc = min(max((y+bh/2)/H, 0.0), 1.0)
                wn = min(max(bw/W, 0.0), 1.0)
                hn = min(max(bh/H, 0.0), 1.0)
                cls = 1 if ann.get('category_id_3')==0 else 0
                lines.append(f'{cls} {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}')
            (yolo_out_dir/'labels'/split/(src_path.stem+'.txt')).write_text('\n'.join(lines))

    yaml = f"""path: {yolo_out_dir}
train: images/train
val:   images/val
test:  images/test
nc: 2
names:
  0: erupted
  1: impacted
"""
    (yolo_out_dir/'dataset.yaml').write_text(yaml)
    return splits, errors

# Generar split base si no existe
def split_existe(d):
    return d.exists() and all(
        len(list((d/'images'/s).glob('*.*')))>0 for s in ['train','val','test'])

if split_existe(YOLO_DIR):
    log('Split YOLO base encontrado', 'OK')
else:
    log('Generando split YOLO base...', 'WARN')
    img_ids = list(wisdom_c.keys())
    labels  = [classify_image(i) for i in img_ids]
    splits_base, err = generar_split_yolo(YOLO_DIR, img_ids, labels,
                                          wisdom_c, id_to_image_c, IMG_DIR_C)
    log(f'Split base generado{f" ({err} errores)" if err else ""}', 'OK')

def collate_fn(batch):
    return torch.stack([b[0] for b in batch]), [b[1] for b in batch]

BASE_TRANSFORM = T.Compose([
    T.Resize((640, 640)),
    T.ToTensor(),
    T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

ds_val  = DentexDataset('val',  YOLO_DIR, transform=BASE_TRANSFORM)
ds_test = DentexDataset('test', YOLO_DIR, transform=BASE_TRANSFORM)
BATCH_SIZE = 8
dl_val  = DataLoader(ds_val,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, collate_fn=collate_fn)
dl_test = DataLoader(ds_test, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, collate_fn=collate_fn)

log(f'Particionado 70/5/25 (seed=42, estratificado):', 'DATA')
for split in ['train','val','test']:
    n = len(list((YOLO_DIR/'images'/split).glob('*.*')))
    log(f'  {split}: {n} imagenes', 'DATA')

## Seccion 6 — Preprocesamiento y normalizacion

| Paso | Valor | Justificacion |
|------|-------|---------------|
| `Resize(640,640)` | 640x640 px | Tamano estandar de YOLOv8 |
| `ToTensor()` | float32 [0,1] | Convierte PIL uint8 [0,255] |
| `Normalize(mean, std)` | ImageNet | Backbone de YOLOv8 preentrenado con estas estadisticas |

**Nota sobre escala de grises:**  
Las radiografias son grises convertidas a RGB (3 canales identicos, verificado con PIL).  
Normalizar con ImageNet RGB equivale a normalizar el canal gris con los mismos valores.

In [ ]:
ds_base = DentexDataset('train', YOLO_DIR, transform=BASE_TRANSFORM)
img_raw = ds_base.get_image_raw(0)
img_t, target = ds_base[0]

mean_t = torch.tensor([0.485,0.456,0.406]).view(3,1,1)
std_t  = torch.tensor([0.229,0.224,0.225]).view(3,1,1)

log('Verificacion de preprocesamiento:', 'DATA')
log(f'  PIL:    {img_raw.size}  modo={img_raw.mode}', 'DATA')
log(f'  Tensor: {tuple(img_t.shape)}  dtype={img_t.dtype}', 'DATA')
log(f'  Rango:  [{img_t.min():.3f}, {img_t.max():.3f}]  (normalizado)', 'DATA')

# Verificar que los 3 canales son iguales (escala de grises)
arr = np.array(img_raw)
log(f'  R==G: {np.array_equal(arr[:,:,0], arr[:,:,1])}  R==B: {np.array_equal(arr[:,:,0], arr[:,:,2])}  (grayscale confirmado)', 'DATA')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(img_raw)
axes[0].set_title(f'PIL original — {img_raw.size[0]}x{img_raw.size[1]} px')
axes[0].axis('off')
img_vis = (img_t*std_t+mean_t).clamp(0,1).permute(1,2,0).numpy()
axes[1].imshow(img_vis)
axes[1].set_title('Tensor desnormalizado — 640x640 px')
if len(target['boxes'])>0:
    for box, lbl in zip(target['boxes'], target['labels']):
        xc,yc,bw,bh = box.tolist()
        x1=(xc-bw/2)*640; y1=(yc-bh/2)*640
        color='#FF4444' if lbl==1 else '#44CC44'
        axes[1].add_patch(patches.Rectangle((x1,y1),bw*640,bh*640,linewidth=2,edgecolor=color,facecolor='none'))
axes[1].axis('off')
plt.suptitle('Preprocesamiento: PIL original vs tensor normalizado')
plt.tight_layout()
plt.savefig(str(OUTPUTS_DIR/'preprocesamiento.png'),dpi=100,bbox_inches='tight')
plt.show()
log('Guardado: preprocesamiento.png', 'OK')

## Seccion 7 — Combinaciones de Data Augmentation

Evaluamos 6 combinaciones de augmentation para el conjunto de **train unicamente**.  
Val y test siempre usan `BASE_TRANSFORM` para evaluacion determinista.

| ID | Nombre | Transformaciones |
|----|--------|-----------------|
| A | Baseline | Solo resize + normalize |
| B | Flips | + HorizontalFlip |
| C | Color | + ColorJitter + ligera rotacion |
| D | Geometric | + Affine + RandomErasing |
| E | Full | B + C + D (todo junto) |
| F | MixAug | Full + GaussianBlur (simula distintas calidades de equipo) |

**Justificacion clinica:**
- HorizontalFlip: el arco dental es simetrico, el espejo es anatomicamente valido
- ColorJitter: simula variabilidad de contraste entre los 3 hospitales del dataset
- RandomAffine (leve): variaciones de posicionamiento del paciente en el equipo
- RandomErasing: simula artefactos metalicos que ocluyen partes de la imagen
- GaussianBlur: simula distintas calidades de equipos radiograficos
- **NO** aplicamos VerticalFlip ni rotaciones grandes: anatomicamente incorrectas

In [ ]:
AUGMENTATIONS = {
    'A_baseline': T.Compose([
        T.Resize((640,640)),
        T.ToTensor(),
        T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
    ]),
    'B_flips': T.Compose([
        T.Resize((640,640)),
        T.RandomHorizontalFlip(p=0.5),
        T.ToTensor(),
        T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
    ]),
    'C_color': T.Compose([
        T.Resize((640,640)),
        T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.1),
        T.RandomAffine(degrees=5),
        T.ToTensor(),
        T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
    ]),
    'D_geometric': T.Compose([
        T.Resize((640,640)),
        T.RandomAffine(degrees=8, translate=(0.05,0.05), scale=(0.9,1.1)),
        T.ToTensor(),
        T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
        T.RandomErasing(p=0.15, scale=(0.02,0.1), value='random')
    ]),
    'E_full': T.Compose([
        T.Resize((640,640)),
        T.RandomHorizontalFlip(p=0.5),
        T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.1),
        T.RandomAffine(degrees=5, translate=(0.05,0.05)),
        T.ToTensor(),
        T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
        T.RandomErasing(p=0.1, scale=(0.02,0.1), value='random')
    ]),
    'F_mixaug': T.Compose([
        T.Resize((640,640)),
        T.RandomHorizontalFlip(p=0.5),
        T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.15),
        T.RandomAffine(degrees=5, translate=(0.05,0.05)),
        T.GaussianBlur(kernel_size=3, sigma=(0.1,1.5)),
        T.ToTensor(),
        T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
        T.RandomErasing(p=0.1, scale=(0.02,0.1), value='random')
    ]),
}

log(f'{len(AUGMENTATIONS)} combinaciones de augmentation definidas', 'OK')
for name in AUGMENTATIONS:
    log(f'  {name}', 'DATA')

In [ ]:
# Visualizacion: 4 imagenes x 4 versiones de cada combinacion
indices = random.sample(range(len(ds_base)), 4)
n_versions = 4

for aug_name, aug_transform in AUGMENTATIONS.items():
    ds_aug = DentexDataset('train', YOLO_DIR, transform=aug_transform)
    fig, axes = plt.subplots(4, n_versions+1, figsize=(22, 16))

    for row, idx in enumerate(indices):
        img_raw = ds_aug.get_image_raw(idx)
        axes[row][0].imshow(img_raw.resize((640,640)))
        axes[row][0].set_title('Original' if row==0 else '', fontsize=8)
        axes[row][0].axis('off')
        for col in range(n_versions):
            aug_t, _ = ds_aug[idx]
            img_vis = (aug_t*std_t+mean_t).clamp(0,1).permute(1,2,0).numpy()
            axes[row][col+1].imshow(img_vis)
            axes[row][col+1].set_title(f'Aug {col+1}' if row==0 else '', fontsize=8)
            axes[row][col+1].axis('off')

    plt.suptitle(f'Augmentation {aug_name} — 4 imagenes x {n_versions} versiones', fontsize=12)
    plt.tight_layout()
    out_path = OUTPUTS_DIR / f'aug_{aug_name}.png'
    plt.savefig(str(out_path), dpi=80, bbox_inches='tight')
    plt.show()
    log(f'Guardado: aug_{aug_name}.png', 'OK')

## Seccion 8 — Expansion del dataset con division en cuadrantes

Las radiografias panoramicas son muy anchas (~2800px). Al redimensionar a 640px  
las muelas del juicio quedan pequenas (~30-50px), lo que dificulta la deteccion.

**Estrategia:** dividir cada imagen en 4 cuadrantes solapados y agregarlos como  
nuevas imagenes de entrenamiento, ajustando las anotaciones correspondientes.

Esto multiplica el dataset de train por hasta **5x** (imagen original + 4 cuadrantes)  
y aumenta la resolucion efectiva de las muelas del juicio.

**Proceso:**
1. Para cada imagen de train, calcular los 4 cuadrantes con 10% de solapamiento
2. Para cada cuadrante, filtrar solo las anotaciones que caen dentro de el
3. Recalcular las coordenadas de los boxes relativos al cuadrante
4. Guardar como nuevas imagenes en el dataset expandido

In [ ]:
def expandir_con_cuadrantes(yolo_src, yolo_dst, overlap=0.1, con_flips=False):
    """
    Expande el split de train dividiendo cada imagen en 4 cuadrantes.
    Val y test se copian sin modificar.

    Args:
        yolo_src:  Path al YOLO_DIR original
        yolo_dst:  Path al nuevo directorio expandido
        overlap:   Solapamiento entre cuadrantes (0.0-0.5)
        con_flips: Si True, agrega tambien el flip horizontal de cada cuadrante
    """
    # Copiar val y test sin cambios
    for split in ['val', 'test']:
        for sub in ['images', 'labels']:
            src_dir = yolo_src / sub / split
            dst_dir = yolo_dst / sub / split
            dst_dir.mkdir(parents=True, exist_ok=True)
            for f in src_dir.glob('*.*'):
                shutil.copy2(f, dst_dir / f.name)

    # Expandir train
    (yolo_dst/'images'/'train').mkdir(parents=True, exist_ok=True)
    (yolo_dst/'labels'/'train').mkdir(parents=True, exist_ok=True)

    train_img_dir = yolo_src / 'images' / 'train'
    train_lbl_dir = yolo_src / 'labels' / 'train'

    n_original = 0
    n_cuadrantes = 0

    for img_path in sorted(train_img_dir.glob('*.png')) + sorted(train_img_dir.glob('*.jpg')):
        lbl_path = train_lbl_dir / (img_path.stem + '.txt')
        if not lbl_path.exists(): continue

        pil = Image.open(img_path).convert('RGB')
        W, H = pil.size

        # Copiar imagen original
        shutil.copy2(img_path, yolo_dst/'images'/'train'/img_path.name)
        shutil.copy2(lbl_path, yolo_dst/'labels'/'train'/lbl_path.name)
        n_original += 1

        # Leer boxes originales
        boxes_orig = []
        for line in lbl_path.read_text().strip().split('\n'):
            if line:
                p = line.split()
                boxes_orig.append((int(p[0]), *map(float, p[1:])))

        # Definir 4 cuadrantes con solapamiento
        ox = int(W * overlap / 2)
        oy = int(H * overlap / 2)
        cuadrantes = [
            ('tl', 0,    0,    W//2+ox, H//2+oy),
            ('tr', W//2-ox, 0, W,       H//2+oy),
            ('bl', 0,    H//2-oy, W//2+ox, H),
            ('br', W//2-ox, H//2-oy, W,  H),
        ]

        for quad_name, x1, y1, x2, y2 in cuadrantes:
            qW, qH = x2-x1, y2-y1
            crop = pil.crop((x1, y1, x2, y2))

            # Ajustar boxes al cuadrante
            quad_lines = []
            for cls, xc, yc, bw, bh in boxes_orig:
                # Convertir a coordenadas absolutas de la imagen original
                abs_xc = xc * W
                abs_yc = yc * H
                abs_bw = bw * W
                abs_bh = bh * H
                abs_x1 = abs_xc - abs_bw/2
                abs_y1 = abs_yc - abs_bh/2
                abs_x2 = abs_xc + abs_bw/2
                abs_y2 = abs_yc + abs_bh/2

                # Calcular interseccion con el cuadrante
                ix1 = max(abs_x1, x1) - x1
                iy1 = max(abs_y1, y1) - y1
                ix2 = min(abs_x2, x2) - x1
                iy2 = min(abs_y2, y2) - y1

                if ix2 <= ix1 or iy2 <= iy1: continue  # No hay interseccion

                # Solo incluir si al menos 50% del box esta en el cuadrante
                inter_area = (ix2-ix1) * (iy2-iy1)
                box_area   = abs_bw * abs_bh
                if inter_area / box_area < 0.5: continue

                # Normalizar al cuadrante
                nxc = (ix1+ix2)/2 / qW
                nyc = (iy1+iy2)/2 / qH
                nw  = (ix2-ix1) / qW
                nh  = (iy2-iy1) / qH
                nxc = min(max(nxc,0),1); nyc = min(max(nyc,0),1)
                nw  = min(max(nw, 0),1); nh  = min(max(nh, 0),1)
                quad_lines.append(f'{cls} {nxc:.6f} {nyc:.6f} {nw:.6f} {nh:.6f}')

            if not quad_lines: continue  # No hay muelas en este cuadrante

            # Guardar cuadrante
            stem = f'{img_path.stem}_{quad_name}'
            crop.save(str(yolo_dst/'images'/'train'/(stem+'.png')))
            (yolo_dst/'labels'/'train'/(stem+'.txt')).write_text('\n'.join(quad_lines))
            n_cuadrantes += 1

            # Agregar flip horizontal del cuadrante
            if con_flips:
                crop_flip = crop.transpose(Image.FLIP_LEFT_RIGHT)
                flip_lines = []
                for line in quad_lines:
                    p = line.split()
                    cls2, xc2, yc2, bw2, bh2 = int(p[0]), *map(float, p[1:])
                    flip_lines.append(f'{cls2} {1-xc2:.6f} {yc2:.6f} {bw2:.6f} {bh2:.6f}')
                stem_flip = f'{img_path.stem}_{quad_name}_flip'
                crop_flip.save(str(yolo_dst/'images'/'train'/(stem_flip+'.png')))
                (yolo_dst/'labels'/'train'/(stem_flip+'.txt')).write_text('\n'.join(flip_lines))
                n_cuadrantes += 1

    # Generar yaml
    yaml = f"""path: {yolo_dst}
train: images/train
val:   images/val
test:  images/test
nc: 2
names:
  0: erupted
  1: impacted
"""
    (yolo_dst/'dataset.yaml').write_text(yaml)
    return n_original, n_cuadrantes

log('Generando dataset expandido con cuadrantes...', 'INFO')

# Dataset G: cuadrantes sin flip
YOLO_QUAD = DATA_DIR / 'processed' / 'yolo_quad'
if not split_existe(YOLO_QUAD):
    n_orig, n_quad = expandir_con_cuadrantes(YOLO_DIR, YOLO_QUAD, overlap=0.1, con_flips=False)
    log(f'Dataset G (cuadrantes): {n_orig} originales + {n_quad} cuadrantes = {n_orig+n_quad} total', 'OK')
else:
    n_tr = len(list((YOLO_QUAD/'images'/'train').glob('*.*')))
    log(f'Dataset G ya existe: {n_tr} imagenes en train', 'OK')

# Dataset H: cuadrantes + flips
YOLO_QUAD_FLIP = DATA_DIR / 'processed' / 'yolo_quad_flip'
if not split_existe(YOLO_QUAD_FLIP):
    n_orig, n_quad = expandir_con_cuadrantes(YOLO_DIR, YOLO_QUAD_FLIP, overlap=0.1, con_flips=True)
    log(f'Dataset H (cuadrantes+flips): {n_orig} originales + {n_quad} cuadrantes/flips = {n_orig+n_quad} total', 'OK')
else:
    n_tr = len(list((YOLO_QUAD_FLIP/'images'/'train').glob('*.*')))
    log(f'Dataset H ya existe: {n_tr} imagenes en train', 'OK')

In [ ]:
# Visualizar cuadrantes con sus boxes ajustadas
log('Verificacion visual de cuadrantes con boxes ajustadas...', 'INFO')

# Tomar una imagen de train que tenga muelas en varios cuadrantes
train_imgs_base = list((YOLO_DIR/'images'/'train').glob('*.png'))
sample_img = random.choice(train_imgs_base)
sample_lbl = YOLO_DIR/'labels'/'train'/(sample_img.stem+'.txt')

pil_orig = Image.open(sample_img).convert('RGB')
W, H = pil_orig.size

fig, axes = plt.subplots(2, 3, figsize=(20, 10))

# Imagen completa con boxes
ax = axes[0][0]
ax.imshow(np.array(pil_orig))
ax.axhline(y=H//2, color='yellow', linewidth=1.5, linestyle='--')
ax.axvline(x=W//2, color='yellow', linewidth=1.5, linestyle='--')
for line in sample_lbl.read_text().strip().split('\n'):
    if not line: continue
    p = line.split(); cls,xc,yc,bw,bh = int(p[0]),*map(float,p[1:])
    x1=(xc-bw/2)*W; y1=(yc-bh/2)*H
    color='#FF4444' if cls==1 else '#44CC44'
    ax.add_patch(patches.Rectangle((x1,y1),bw*W,bh*H,linewidth=2,edgecolor=color,facecolor='none'))
ax.set_title(f'Original ({W}x{H})')
ax.axis('off')

# 4 cuadrantes con boxes ajustadas
ox = int(W*0.05); oy = int(H*0.05)
cuads = [('tl',0,0,W//2+ox,H//2+oy), ('tr',W//2-ox,0,W,H//2+oy),
         ('bl',0,H//2-oy,W//2+ox,H), ('br',W//2-ox,H//2-oy,W,H)]
positions = [(0,1),(0,2),(1,0),(1,1)]

for (qn,qx1,qy1,qx2,qy2), (r,c) in zip(cuads, positions):
    crop = pil_orig.crop((qx1,qy1,qx2,qy2))
    qW,qH = qx2-qx1, qy2-qy1
    ax = axes[r][c]
    ax.imshow(np.array(crop))

    # Buscar el archivo del cuadrante en el dataset expandido
    quad_lbl = YOLO_QUAD/'labels'/'train'/(sample_img.stem+f'_{qn}.txt')
    if quad_lbl.exists():
        for line in quad_lbl.read_text().strip().split('\n'):
            if not line: continue
            p = line.split(); cls,xc,yc,bw,bh = int(p[0]),*map(float,p[1:])
            x1=(xc-bw/2)*qW; y1=(yc-bh/2)*qH
            color='#FF4444' if cls==1 else '#44CC44'
            ax.add_patch(patches.Rectangle((x1,y1),bw*qW,bh*qH,linewidth=2,edgecolor=color,facecolor='none'))
        ax.set_title(f'Cuadrante {qn} ({qW}x{qH}) — con boxes')
    else:
        ax.set_title(f'Cuadrante {qn} — sin muelas')
    ax.axis('off')

axes[1][2].axis('off')
plt.suptitle('Division en cuadrantes con boxes ajustadas\nAmarillo = linea de division')
plt.tight_layout()
plt.savefig(str(OUTPUTS_DIR/'cuadrantes_verificacion.png'),dpi=100,bbox_inches='tight')
plt.show()
log('Guardado: cuadrantes_verificacion.png', 'OK')

## Seccion 9 — Verificacion final del batch

In [ ]:
ds_train_full = DentexDataset('train', YOLO_DIR, transform=AUGMENTATIONS['E_full'])
dl_train = DataLoader(ds_train_full, batch_size=BATCH_SIZE, shuffle=True,
                      num_workers=2, collate_fn=collate_fn, pin_memory=True)

batch_imgs, batch_targets = next(iter(dl_train))

log('Verificacion del batch (augmentation E_full):', 'DATA')
log(f'  Shape:            {tuple(batch_imgs.shape)}  (B x C x H x W)', 'DATA')
log(f'  Dtype:            {batch_imgs.dtype}', 'DATA')
log(f'  Rango:            [{batch_imgs.min():.3f}, {batch_imgs.max():.3f}]', 'DATA')
log(f'  Boxes por imagen: {[len(t["boxes"]) for t in batch_targets]}', 'DATA')

assert not torch.isnan(batch_imgs).any(), 'NaN detectado'
assert not torch.isinf(batch_imgs).any(), 'Inf detectado'
log('Sin NaN ni Inf', 'OK')

n_show = min(8, len(batch_targets))
fig, axes = plt.subplots(2, 4, figsize=(20,8))
for idx, ax in enumerate(axes.flatten()):
    if idx >= n_show: ax.axis('off'); continue
    img_t  = batch_imgs[idx]
    target = batch_targets[idx]
    img_vis = (img_t*std_t+mean_t).clamp(0,1).permute(1,2,0).numpy()
    ax.imshow(img_vis)
    for box, lbl in zip(target['boxes'], target['labels']):
        xc,yc,bw,bh = box.tolist()
        x1=(xc-bw/2)*640; y1=(yc-bh/2)*640
        color='#FF4444' if lbl==1 else '#44CC44'
        ax.add_patch(patches.Rectangle((x1,y1),bw*640,bh*640,linewidth=2,edgecolor=color,facecolor='none'))
        ax.text(x1+2,y1-8,'imp' if lbl==1 else 'eru',color='white',fontsize=7,fontweight='bold',
                bbox=dict(facecolor=color,alpha=0.8,pad=1,edgecolor='none'))
    ax.set_title(f'{target["image_id"]}  {len(target["boxes"])} boxes',fontsize=7)
    ax.axis('off')
plt.suptitle('Batch verificado — desnormalizado con boxes\nVerde=erupted  Rojo=impacted')
plt.tight_layout()
plt.savefig(str(OUTPUTS_DIR/'batch_verificacion.png'),dpi=100,bbox_inches='tight')
plt.show()
log('Guardado: batch_verificacion.png', 'OK')

## Seccion 10 — Exportar CSVs para GitHub

In [ ]:
for split in ['train','val','test']:
    rows = []
    img_dir = YOLO_DIR/'images'/split
    lbl_dir = YOLO_DIR/'labels'/split
    for img_path in sorted(list(img_dir.glob('*.png'))+list(img_dir.glob('*.jpg'))):
        lbl_path = lbl_dir/(img_path.stem+'.txt')
        if not lbl_path.exists(): continue
        classes = [int(l.split()[0]) for l in lbl_path.read_text().strip().split('\n') if l]
        rows.append({
            'filename':   img_path.name,
            'split':      split,
            'img_class':  'impacted' if 1 in classes else 'erupted',
            'n_boxes':    len(classes),
            'n_erupted':  classes.count(0),
            'n_impacted': classes.count(1),
        })
    df = pd.DataFrame(rows)
    csv_path = SPLITS_DIR/f'{split}.csv'
    df.to_csv(csv_path, index=False)
    log(f'CSV {split}: {len(df)} filas → {csv_path}', 'OK')

# Descargar en Colab
if IN_COLAB:
    from google.colab import files
    for split in ['train','val','test']:
        files.download(str(SPLITS_DIR/f'{split}.csv'))
    log('CSVs descargados — subir a data/ en el repo de GitHub', 'OK')

## Seccion 11 — HierarchicalDet: arquitectura del paper

### Arquitectura

El metodo baseline del paper es **HierarchicalDet**, que combina dos ideas:

**1. DiffusionDet** (base)
- Reformula la deteccion de objetos como un proceso de difusion: parte de boxes ruidosas aleatorias y las "desruiza" iterativamente hasta llegar a las detecciones finales
- Usa un encoder visual (ResNet/Swin Transformer) + decoder de difusion
- Ventaja: maneja bien la incertidumbre en la localizacion de objetos

**2. Aprendizaje jerarquico** (novedad del paper)
- Entrena en 3 etapas siguiendo la jerarquia FDI:
  - Etapa 1: solo dataset (a) → aprende cuadrantes
  - Etapa 2: transfiere pesos, entrena con dataset (b) → aprende enumeracion
  - Etapa 3: transfiere pesos, entrena con dataset (c) → aprende diagnostico
- Usa "noisy box manipulation": en cada etapa, inicializa los boxes ruidosos con las predicciones de la etapa anterior
- Permite aprender de anotaciones parciales (datasets a y b) para mejorar la tarea final

### Por que no lo implementamos directamente

| Razon | Detalle |
|-------|---------|
| **detectron2** | Framework de Meta requerido. Instalacion compleja, incompatible con versiones actuales de PyTorch/CUDA en Colab |
| **Inferencia lenta** | DiffusionDet es iterativo (~10s/imagen en GPU). Poco practico para una app web |
| **Tarea distinta** | Disenado para 3 jerarquias simultaneas. Nuestro caso es deteccion binaria |
| **Sin script simplificado** | El repo oficial no tiene fine-tuning directo. Requiere adaptar todo el pipeline de detectron2 |
| **Memoria** | Requiere >16GB VRAM para entrenamiento. Colab T4 tiene 15GB |

### Lo que si tomamos del paper
- Las metricas de evaluacion: AP50, AP75, mAP (COCO-style) → YOLOv8 las calcula automaticamente
- La idea de usar los 3 datasets jerarquicamente → implementamos una version simplificada abajo
- Referencia de resultados para comparar nuestro modelo

In [ ]:
# Implementacion parcial de HierarchicalDet:
# Cargamos los 3 datasets jerarquicos y mostramos su estructura
# No podemos correr el modelo completo, pero si preparar los datos como lo haria el paper

log('Cargando los 3 datasets jerarquicos del paper...', 'INFO')

# Dataset (a): solo cuadrante
json_a = next((j for j in json_files if 'quadrant' in j.name.lower()
               and 'enumeration' not in j.name.lower()
               and 'disease' not in j.name.lower()), None)

# Dataset (b): cuadrante + enumeracion
json_b = next((j for j in json_files if 'enumeration' in j.name.lower()
               and 'disease' not in j.name.lower()), None)

# Dataset (c): completo — ya lo tenemos como TRAIN_JSON
json_c = TRAIN_JSON

for name, j in [('(a) cuadrante', json_a), ('(b) enumeracion', json_b), ('(c) diagnostico', json_c)]:
    if j and is_real_file(j):
        with open(j) as f:
            d = json.load(f)
        log(f'Dataset {name}: {len(d["images"])} imagenes, {len(d["annotations"])} anotaciones', 'DATA')
    else:
        log(f'Dataset {name}: no encontrado', 'WARN')

log('', 'INFO')
log('En HierarchicalDet el entrenamiento seria:', 'INFO')
log('  Etapa 1: entrenar DiffusionDet con dataset (a) — detecta cuadrantes', 'INFO')
log('  Etapa 2: cargar pesos de etapa 1, entrenar con dataset (b) — detecta dientes', 'INFO')
log('  Etapa 3: cargar pesos de etapa 2, entrenar con dataset (c) — detecta patologias', 'INFO')
log('  Cada etapa inicializa los boxes ruidosos con las predicciones de la anterior', 'INFO')
log('', 'INFO')
log('Nuestro enfoque (YOLOv8s desde COCO) es equivalente a la Etapa 3 directamente,', 'INFO')
log('aprovechando que COCO preentrenado ya tiene representaciones de objetos similares.', 'INFO')

## Seccion 12 — Experimentos de fine-tuning

Definimos todos los experimentos. Cada uno combina un **dataset** con una **augmentation**.  
El entrenamiento real se corre en la siguiente entrega.

| Experimento | Dataset | Augmentation | Objetivo |
|-------------|---------|--------------|----------|
| Exp1 | Base (440 imgs) | A: Baseline | Referencia sin augmentation |
| Exp2 | Base (440 imgs) | B: Flips | Solo flips horizontales |
| Exp3 | Base (440 imgs) | E: Full | Augmentation completa |
| Exp4 | Base (440 imgs) | F: MixAug | Full + GaussianBlur |
| Exp5 | Quad (~1500 imgs) | A: Baseline | Solo expansion por cuadrantes |
| Exp6 | Quad+Flip (~2500 imgs) | B: Flips | Maxima expansion del dataset |
| Exp7 | Quad+Flip (~2500 imgs) | E: Full | Expansion + augmentation completa |

**Hipotesis:**  
Exp5-7 deberian superar a Exp1-4 porque las muelas quedan mas grandes en los cuadrantes.  
Exp7 deberia ser el mejor al combinar mas datos con augmentation agresiva.

In [ ]:
from ultralytics import YOLO

EXPERIMENTOS = {
    'Exp1_base_noaug':    {'data': YOLO_DIR,       'augment': False, 'fliplr': 0.0},
    'Exp2_base_flips':    {'data': YOLO_DIR,       'augment': True,  'fliplr': 0.5},
    'Exp3_base_full':     {'data': YOLO_DIR,       'augment': True,  'fliplr': 0.5,
                           'hsv_h':0.02,'hsv_s':0.5,'hsv_v':0.3,'degrees':5,'translate':0.05},
    'Exp4_base_mixaug':   {'data': YOLO_DIR,       'augment': True,  'fliplr': 0.5,
                           'hsv_h':0.02,'hsv_s':0.6,'hsv_v':0.4,'degrees':5,'blur_p':0.1},
    'Exp5_quad_noaug':    {'data': YOLO_QUAD,      'augment': False, 'fliplr': 0.0},
    'Exp6_quadflip_flips':{'data': YOLO_QUAD_FLIP, 'augment': True,  'fliplr': 0.5},
    'Exp7_quadflip_full': {'data': YOLO_QUAD_FLIP, 'augment': True,  'fliplr': 0.5,
                           'hsv_h':0.02,'hsv_s':0.5,'hsv_v':0.3,'degrees':5,'translate':0.05},
}

log('Experimentos definidos:', 'DATA')
for name, cfg in EXPERIMENTOS.items():
    n_train = len(list((cfg['data']/'images'/'train').glob('*.*')))
    log(f'  {name}: {n_train} imgs train | data={cfg["data"].name}', 'DATA')

log('', 'INFO')
log('Para correr un experimento:', 'INFO')
log('  model = YOLO("yolov8s.pt")', 'INFO')
log('  model.train(data=str(cfg["data"]/"dataset.yaml"), epochs=100,', 'INFO')
log('              imgsz=640, batch=8, patience=20, **cfg)', 'INFO')

In [ ]:
# Correr Exp1 como prueba de que el setup funciona (5 epochs)
# Descomentar para correr el experimento completo (100 epochs)

if torch.cuda.is_available():
    log('Corriendo Exp1 (5 epochs de prueba)...', 'INFO')
    model = YOLO('yolov8s.pt')
    results = model.train(
        data=str(YOLO_DIR/'dataset.yaml'),
        epochs=5,          # cambia a 100 para el experimento real
        imgsz=640,
        batch=8,
        patience=20,
        fliplr=0.0,
        augment=False,
        name='Exp1_base_noaug',
        exist_ok=True,
        verbose=False,
    )
    log(f'Exp1 completado — mAP50: {results.results_dict.get("metrics/mAP50(B)", "N/A")}', 'OK')
else:
    log('Sin GPU — saltando entrenamiento de prueba', 'WARN')
    log('Conectar GPU y correr esta celda para verificar el setup', 'INFO')

## Seccion 13 — Conclusiones

### Resumen de decisiones

**Dataset:**
- 440 imagenes con muela del juicio de 705 disponibles (cat2==7)
- Split estratificado 70/5/25 con seed=42
- Dataset expandido con cuadrantes: ~1500-2500 imagenes segun configuracion

**Preprocesamiento:**
- Resize 640x640 (tamano YOLOv8)
- Normalizacion ImageNet (backbone preentrenado en COCO)

**Augmentation elegida para produccion:** E_full (HFlip + ColorJitter + Affine + RandomErasing)
- Justificacion clinica para cada transformacion
- Solo aplicada a train

**Modelo:** YOLOv8s fine-tuning desde COCO
- Descarta HierarchicalDet por complejidad de implementacion e incompatibilidad con Colab
- YOLOv8 tiene pipeline de augmentation interno (mosaic, mixup) que complementa el nuestro

**Experimentos planificados:**
- 7 combinaciones dataset x augmentation
- Hipotesis: expansion con cuadrantes + augmentation completa = mejor resultado
- Metricas a reportar: mAP50, mAP50-95, AP erupted, AP impacted

### Proximos pasos (Entrega 3)
1. Correr los 7 experimentos completos (100 epochs cada uno)
2. Comparar metricas en test set
3. Seleccionar el mejor modelo
4. Implementar la app Streamlit

In [ ]:
log('='*50, 'INFO')
log('RESUMEN ENTREGA 2', 'INFO')
log('='*50, 'INFO')
log(f'Dataset base:          {len(ds_train_full)} imgs train', 'DATA')
log(f'Dataset quad:          {len(list((YOLO_QUAD/"images"/"train").glob("*.*")))} imgs train', 'DATA')
log(f'Dataset quad+flip:     {len(list((YOLO_QUAD_FLIP/"images"/"train").glob("*.*")))} imgs train', 'DATA')
log(f'Val:                   {len(ds_val)} imgs', 'DATA')
log(f'Test:                  {len(ds_test)} imgs', 'DATA')
log(f'Augmentations:         {len(AUGMENTATIONS)} combinaciones definidas', 'DATA')
log(f'Experimentos:          {len(EXPERIMENTOS)} configurados', 'DATA')
log(f'Modelo:                YOLOv8s desde COCO', 'DATA')
log(f'Outputs:               {OUTPUTS_DIR}', 'OK')
log(f'CSVs:                  {SPLITS_DIR}', 'OK')